[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/04_machine_learning/15_model_evaluation.ipynb)

# 📓 Notebook 15 — Model Evaluation Deep Dive

> **Module:** Machine Learning · **Estimated time:** 60–75 min · **Difficulty:** Intermediate

In Notebook 14 you trained your first models with scikit-learn and looked at accuracy. That's enough to get started; it is not enough to ship. Real model evaluation is about **picking the right metric for the cost structure**, understanding **trade-offs**, and producing **honest** numbers that survive contact with a stakeholder.

This notebook is a tour of the evaluation toolkit every working ML engineer carries:

- The confusion matrix (and what it implies for cost).
- Precision / recall / F1, and the **threshold trade-off**.
- ROC and PR curves — when each one matters.
- **Calibration** — when probabilities lie.
- Cross-validation done properly.
- Learning curves — diagnose under- vs over-fitting.

We use the churn dataset from NB 14 so you can compare against the metrics you saw there.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Read a confusion matrix and explain its four cells in *cost units*, not just counts.
2. Pick between **precision**, **recall**, and **F1** based on which error is more expensive.
3. Slide the **decision threshold** to trade precision against recall — and visualise the trade-off.
4. Plot and interpret **ROC** and **precision-recall** curves.
5. Diagnose whether a model's **probabilities are calibrated** — and recalibrate them if not.
6. Use **cross-validation** correctly, including stratified folds and the `Pipeline` discipline.
7. Read a **learning curve** to decide whether more data would help.

## ✅ Prerequisites

NB 14 (sklearn basics), NB 10 (statistics — confidence intervals).

## 1. Setup — load the same churn dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose       import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline      import Pipeline
from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble      import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, learning_curve
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_recall_curve, roc_curve, auc, brier_score_loss,
                              RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.model_selection import LearningCurveDisplay
from sklearn.calibration import calibration_curve, CalibratedClassifierCV, CalibrationDisplay

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)
RANDOM_STATE = 42

**Recreate the synthetic churn dataset from NB 14, then split**

In [ ]:
# Recreate the synthetic churn dataset from NB 14
rng = np.random.default_rng(RANDOM_STATE)
n = 600
mrr_eur         = rng.lognormal(mean=6.5, sigma=0.7, size=n).clip(50, 50_000)
months_active   = rng.integers(1, 60, size=n)
support_tickets = rng.poisson(lam=2.5, size=n)
plan            = rng.choice(["Free","Standard","Premium","Enterprise"],
                              size=n, p=[0.15,0.45,0.30,0.10])
last_login_days = rng.integers(0, 90, size=n)
plan_adj = np.array([{"Free":-0.45,"Standard":0.0,"Premium":0.15,"Enterprise":0.35}[p]
                      for p in plan])
happiness = (-0.40*(support_tickets-2.5)/2.0
             -0.80*(last_login_days-45)/30.0
             +0.35*(np.log1p(mrr_eur)-6.5)/1.5
             +0.30*(months_active-30)/20.0
             + plan_adj + rng.normal(0,0.30,n))
nps = (40 + 35*happiness + rng.normal(0,8,n)).clip(-100,100)
churn_prob = 1/(1+np.exp(-(-1.6 - 2.5*happiness)))
churned = (rng.random(n) < churn_prob).astype(int)

df = pd.DataFrame({"mrr_eur":mrr_eur, "months_active":months_active,
                    "support_tickets":support_tickets, "nps":nps, "plan":plan,
                    "last_login_days":last_login_days, "churned":churned})

X = df.drop(columns=["churned"])
y = df["churned"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20,
                                           random_state=RANDOM_STATE, stratify=y)

**Build the preprocessing ColumnTransformer**

In [ ]:
prep = ColumnTransformer([
    ("num", StandardScaler(),
        ["mrr_eur","months_active","support_tickets","nps","last_login_days"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["plan"]),
])

print(f"Train: {X_tr.shape}, churn rate {y_tr.mean():.1%}")
print(f"Test : {X_te.shape}, churn rate {y_te.mean():.1%}")

## 2. Two models — same dataset, different probabilities

In [ ]:
logreg = Pipeline([("prep", prep),
                    ("model", LogisticRegression(max_iter=500, random_state=RANDOM_STATE))])
rforest = Pipeline([("prep", prep),
                    ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))])

logreg.fit(X_tr, y_tr)
rforest.fit(X_tr, y_tr)

# Probabilities for the positive (churn) class
p_lr = logreg.predict_proba(X_te)[:, 1]
p_rf = rforest.predict_proba(X_te)[:, 1]

print(f"LR  accuracy: {logreg.score(X_te, y_te):.3f}")
print(f"RF  accuracy: {rforest.score(X_te, y_te):.3f}")


## 3. The confusion matrix — read it in **cost units**

```
                 Predicted: stayed     Predicted: churned
   Actual: stayed    TN                  FP   (false alarm: pestered a happy customer)
   Actual: churned   FN                  TP   (correctly caught the churner)
```

The metric you optimise depends on **which mistake is more expensive**.

In [ ]:
y_pred_lr = (p_lr >= 0.5).astype(int)
cm = confusion_matrix(y_te, y_pred_lr)

# Annotate each quadrant with its name AND its count
quadrants = np.array([[f"TN\n{cm[0, 0]}",                  f"FP (false alarm)\n{cm[0, 1]}"],
                      [f"FN (missed churner)\n{cm[1, 0]}", f"TP\n{cm[1, 1]}"]])
sns.heatmap(cm, annot=quadrants, fmt="", cmap="Blues",
            xticklabels=["stayed", "churned"], yticklabels=["stayed", "churned"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion matrix — Logistic Regression  (threshold = 0.5)")
plt.show()

# Cost example
COST_FP = 5      # cost of pestering a happy customer (e.g., $5 of retention-team time)
COST_FN = 200    # cost of losing a churner (lifetime value)
total_cost = cm[0, 1] * COST_FP + cm[1, 0] * COST_FN
print(f"\nWith FP cost = ${COST_FP}, FN cost = ${COST_FN}: total cost = ${total_cost:,}")

> 🎯 **The cost asymmetry is what changes which model wins.** A model with worse accuracy can still be the right choice if its mistakes are the *cheap* mistakes.

## 4. Precision, recall, F1 — three numbers, one trade-off

- **Precision** = of the items we flagged as churn, how many really churned? *(High precision = few false alarms.)*
- **Recall** = of the items that really churned, how many did we flag? *(High recall = we caught most of them.)*
- **F1** = harmonic mean of the two — penalises imbalance between them.

In [ ]:
print(classification_report(y_te, y_pred_lr,
                             target_names=["stayed", "churned"],
                             digits=3, zero_division=0))


**Reading the report:**

- Precision for "churned" — of everyone the model flagged as a churner, the fraction that really churned. High precision = few false alarms.
- Recall for "churned" is the "catch rate" — what fraction of actual churners we caught.
- The **macro avg** is an unweighted mean over classes (treats both equally — useful for imbalanced data).
- The **weighted avg** weights by class frequency (so the majority class dominates).

> ⚠️ **For imbalanced data, `accuracy` lies.** If 99% of users don't churn, predicting "stay" for everyone scores 99% accuracy and zero recall — a useless model.

## 5. The threshold trade-off — precision vs recall curves

The 0.5 threshold is *not magic*. Slide it lower to catch more churners (higher recall, more false alarms); slide it higher to be more conservative.

In [ ]:
PrecisionRecallDisplay.from_predictions(y_te, p_lr, name="Logistic Regression")
ax = plt.gca()

# Mark two possible operating points on the curve
prec, rec, thr = precision_recall_curve(y_te, p_lr)
for t, marker, color in [(0.5, "o", "red"), (0.2, "s", "orange")]:
    i = np.argmin(np.abs(thr - t))
    ax.plot(rec[i], prec[i], marker, color=color, markersize=10,
            label=f"threshold = {t}: precision {prec[i]:.2f}, recall {rec[i]:.2f}")

ax.set_title("Precision vs Recall — pick your operating point")
ax.legend(loc="lower left");

**Pick the operating point by cost.** If a missed churner costs you $200 and a false alarm costs $5, pick the threshold where the **expected cost is minimised** — usually a *low* threshold (high recall, mediocre precision).

In [ ]:
# Cost as a function of threshold
thresholds = np.linspace(0.05, 0.95, 91)
costs = []
for t in thresholds:
    yp = (p_lr >= t).astype(int)
    cm_ = confusion_matrix(y_te, yp)
    cost = cm_[0,1]*COST_FP + cm_[1,0]*COST_FN
    costs.append(cost)

best_idx = int(np.argmin(costs))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, costs, lw=2, color="#C44E52")
ax.axvline(thresholds[best_idx], color="black", ls=":")
ax.set_xlabel("Threshold");  ax.set_ylabel("Total cost ($)")
ax.set_title(f"Cost-optimal threshold = {thresholds[best_idx]:.2f}  "
             f"(min cost = ${costs[best_idx]:,})")
plt.tight_layout(); plt.show()


### The payoff — the confusion matrix at the cost-optimal threshold

The cost curve told us *where* the minimum is; now let's **see what moving there does**. Below is the confusion matrix at the default 0.5 beside the one at the cost-optimal threshold, both annotated in the same cost units. Watch the bottom-left cell: lowering the threshold converts expensive *missed churners* (FN × \$200) into cheap *false alarms* (FP × \$5) — which is the entire point of tuning it.


In [ ]:
t_opt = thresholds[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, t, title in [(axes[0], 0.5, "Default threshold 0.50"),
                     (axes[1], t_opt, f"Cost-optimal threshold {t_opt:.2f}")]:
    yp = (p_lr >= t).astype(int)
    m = confusion_matrix(y_te, yp)
    cost = m[0, 1] * COST_FP + m[1, 0] * COST_FN
    quad = np.array([[f"TN\n{m[0,0]}",                  f"FP (false alarm)\n{m[0,1]}"],
                     [f"FN (missed churner)\n{m[1,0]}", f"TP\n{m[1,1]}"]])
    sns.heatmap(m, annot=quad, fmt="", cmap="Blues", cbar=False,
                xticklabels=["stayed", "churned"], yticklabels=["stayed", "churned"], ax=ax)
    rec = m[1, 1] / (m[1, 0] + m[1, 1])
    ax.set(xlabel="Predicted", ylabel="Actual",
           title=f"{title}\nrecall {rec:.0%}  ·  total cost ${cost:,}")
fig.suptitle("Lowering the threshold trades cheap false alarms for expensive misses", y=1.03)
plt.tight_layout(); plt.show()

base = confusion_matrix(y_te, (p_lr >= 0.5).astype(int))
opt  = confusion_matrix(y_te, (p_lr >= t_opt).astype(int))
saved = (base[0,1]*COST_FP + base[1,0]*COST_FN) - (opt[0,1]*COST_FP + opt[1,0]*COST_FN)
print(f"Moving 0.50 → {t_opt:.2f} saves ${saved:,} on this test set "
      f"({base[1,0]}→{opt[1,0]} missed churners, {base[0,1]}→{opt[0,1]} false alarms).")


> 🎯 **The threshold is a business decision, not a model property.** Always pick it on a held-out set with your real cost numbers.

## 6. ROC and PR curves — which to look at when

- **ROC** (Receiver Operating Characteristic) plots TPR vs FPR as the threshold changes. Insensitive to class imbalance.
- **PR** (Precision-Recall) plots precision vs recall. More informative for *highly imbalanced* problems (rare positives).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, p in [("LR", p_lr), ("RF", p_rf)]:
    RocCurveDisplay.from_predictions(y_te, p, name=name, ax=axes[0])          # AUC in the legend
    PrecisionRecallDisplay.from_predictions(y_te, p, name=name, ax=axes[1])   # AP in the legend

axes[0].plot([0, 1], [0, 1], "k:", alpha=0.5)   # chance line
axes[0].set_title("ROC curve — LR vs RF")
axes[1].set_title("Precision-Recall curve")
fig.tight_layout();

**How to choose between them in real life:**

| Situation | Curve to look at |
|---|---|
| Balanced classes, two models, comparing overall ranking ability | ROC + AUC |
| Imbalanced classes (rare positives) | PR curve |
| You need a single number for a one-pager | ROC AUC if balanced, PR AUC if imbalanced |
| The cost ratio is wildly skewed | Plot total *cost* as a function of threshold (§5) |

## 7. Calibration — when probabilities lie

A model outputs `P(churn) = 0.8` for a customer. Is this customer **80% likely** to churn — or did the model just *rank* them in the top 20%?

**Well-calibrated** models satisfy: of all customers predicted at 0.8, roughly 80% actually churn. Many tree-based models are *uncalibrated* by default — they're great at ranking but their probabilities are off.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, p in [("LR", p_lr), ("RF (uncalibrated)", p_rf)]:
    CalibrationDisplay.from_predictions(y_te, p, n_bins=10, strategy="quantile", name=name, ax=ax)
    print(f"{name:<20}  Brier score = {brier_score_loss(y_te, p):.3f}  (lower is better)")

ax.set_title("Reliability diagram");   # the dotted diagonal = perfect calibration, drawn for free

**Reading the reliability diagram:**

- Points on the dotted diagonal = perfectly calibrated.
- Points *below* the diagonal = over-confident (model says 0.9 but only 0.7 churn).
- Points *above* the diagonal = under-confident.

> 💡 **The Brier score** is the mean squared error between predicted probabilities and actual outcomes. Lower is better. It's the right summary number for calibration quality.

### Fix it: post-hoc calibration

`CalibratedClassifierCV` wraps any classifier and fits a small monotonic mapping between raw scores and true probabilities. For Random Forests this typically halves the Brier score.

In [ ]:
rf_cal = CalibratedClassifierCV(rforest, method="isotonic", cv=5)
rf_cal.fit(X_tr, y_tr)
p_rf_cal = rf_cal.predict_proba(X_te)[:, 1]

fig, ax = plt.subplots(figsize=(7, 5))
CalibrationDisplay.from_predictions(y_te, p_rf_cal, n_bins=10, strategy="quantile",
                                    name="RF (calibrated, isotonic)", ax=ax)
CalibrationDisplay.from_predictions(y_te, p_rf, n_bins=10, strategy="quantile",
                                    name="RF (uncalibrated)", ax=ax)
ax.set_title("Calibration improvement after CalibratedClassifierCV")
plt.show()

print(f"Brier — uncalibrated RF: {brier_score_loss(y_te, p_rf):.3f}")
print(f"Brier — calibrated   RF: {brier_score_loss(y_te, p_rf_cal):.3f}")

## 8. Cross-validation — the honest evaluation

A single train/test split is *one sample* of how the model performs. Cross-validation gives you a distribution.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(logreg, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"5-fold ROC AUC: {scores.round(3)}")
print(f"Mean ± std    : {scores.mean():.3f} ± {scores.std():.3f}")


**Three rules for honest CV:**

1. **Stratify** on the target for classification — keeps the class balance the same in every fold.
2. **Put preprocessing inside the `Pipeline`** — the scaler/encoder is fit on each fold's training portion only. This is how you avoid *data leakage*.
3. **Report mean ± std**, not just the mean. A model with 0.85 ± 0.02 is much more trustworthy than 0.85 ± 0.12.

### 🔬 What actually happens inside k-fold cross-validation?

The cell above called `cross_val_score(...)` and got back 5 numbers. That looks like magic — *where did 5 scores come from a single dataset?* Here is the whole mechanism, no black box.

**The idea in one sentence:** split the rows into **k equal slices (folds)**; then run **k separate train/test experiments**, each time holding out a *different* fold as the test set and training on the other k−1. Every row gets to be in the test set **exactly once**.

With `k = 5` and 10 rows, the rotation looks like this — `T` marks the held-out **test** fold, `.` marks rows used for **training**:

```text
rows:        0  1 | 2  3 | 4  5 | 6  7 | 8  9      (5 folds of 2 rows each)
            ─────────────────────────────────────
 fold 0  →   T  T | .  . | .  . | .  . | .  .   →  test on rows {0,1}, train on the rest  → score₀
 fold 1  →   .  . | T  T | .  . | .  . | .  .   →  test on rows {2,3}, train on the rest  → score₁
 fold 2  →   .  . | .  . | T  T | .  . | .  .   →  test on rows {4,5}, train on the rest  → score₂
 fold 3  →   .  . | .  . | .  . | T  T | .  .   →  test on rows {6,7}, train on the rest  → score₃
 fold 4  →   .  . | .  . | .  . | .  . | T  T   →  test on rows {8,9}, train on the rest  → score₄
            ─────────────────────────────────────
                                    final score = mean(score₀ … score₄)
```

The **test fold rotates down the diagonal**. Five fits, five held-out evaluations, one averaged number. That's *all* `cross_val_score` is doing.

### A single split is *one noisy sample* — CV averages the noise away

Why bother with five fits when one train/test split already gives a score? Because **one split is one roll of the dice.** Which 20% of rows happened to land in your test set is luck:

```text
   one split  →  ┌──────────────── train (80%) ────────────────┐ ┌─ test (20%) ─┐
                 │  the model never sees the test rows          │ │  ONE score   │
                 └──────────────────────────────────────────────┘ └──────────────┘
                   a "lucky" test slice  → flattering score
                   an "unlucky" slice    → pessimistic score
                   ...and you can't tell which one you got.
```

A *lucky* split (easy rows in the test set) flatters the model; an *unlucky* split punishes it. With one number you have **no idea how much of it is signal vs. luck**.

Cross-validation fixes this by rotating the test slice over the *whole* dataset and reporting the **mean ± std** of the k scores:

- the **mean** is a less noisy estimate (averaging cancels the per-split luck),
- the **std** tells you how *stable* the model is — `0.85 ± 0.02` is trustworthy, `0.85 ± 0.12` is a coin-flip dressed up as a number.

> 🧠 **Mental model.** Cross-validation **reuses scarce data by rotating which slice is the test set.** Instead of paying for one held-out test set (and throwing those rows away from training), every row pulls double duty: it trains in k−1 folds and tests in exactly one. You get k honest evaluations out of the same data.

### 🧪 Proof — a hand-written k-fold loop that *matches* `cross_val_score`

Claims are cheap. Let's **build k-fold by hand** on a tiny inline dataset, print each fold's train/test indices and score, then assert that our manual mean equals sklearn's `cross_val_score` to the last decimal. We hand both the manual loop **and** sklearn the *same* `KFold` splitter, so if our understanding is right, the numbers must agree exactly.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --- a tiny, self-contained, seeded dataset (20 rows, 2 features) ---
rng_dd = np.random.default_rng(0)
X_dd = rng_dd.normal(size=(20, 2))
# a learnable target: positive when feature-0 + feature-1 > 0, with a little noise
y_dd = ((X_dd[:, 0] + X_dd[:, 1] + 0.3 * rng_dd.normal(size=20)) > 0).astype(int)

# ONE splitter object, shared by the manual loop AND sklearn -> identical folds
kf = KFold(n_splits=5, shuffle=True, random_state=0)

def make_model():
    # fresh, unfitted estimator each fold (no state leaks between folds)
    return LogisticRegression()

**The manual k-fold loop — train on k-1 folds, test on the held-out one**

In [ ]:
# --- the manual k-fold loop: train on k-1 folds, test on the held-out one ---
manual_scores = []
print("fold |                  train indices                  |   test indices   | score")
print("-" * 86)
for i, (train_idx, test_idx) in enumerate(kf.split(X_dd)):
    model = make_model()
    model.fit(X_dd[train_idx], y_dd[train_idx])          # train on k-1 folds
    preds = model.predict(X_dd[test_idx])                # predict the held-out fold
    score = accuracy_score(y_dd[test_idx], preds)        # score this fold
    manual_scores.append(score)
    tr = [int(x) for x in train_idx]                     # plain ints -> tidy printing
    te = [int(x) for x in test_idx]
    print(f"  {i}  | {str(tr):<47} | {str(te):<16} | {score:.3f}")

manual_scores = np.array(manual_scores)
print("-" * 86)
print(f"manual fold scores : {manual_scores.round(3)}")
print(f"manual mean        : {manual_scores.mean():.6f}")

**Compare to sklearn's cross_val_score on the same splitter, then prove they match**

In [ ]:
# --- sklearn's cross_val_score, given the SAME splitter ---
sklearn_scores = cross_val_score(make_model(), X_dd, y_dd, cv=kf, scoring="accuracy")
print(f"sklearn fold scores: {sklearn_scores.round(3)}")
print(f"sklearn mean       : {sklearn_scores.mean():.6f}")

# --- the proof: identical, fold-by-fold AND on the mean ---
assert np.allclose(manual_scores, sklearn_scores), "per-fold scores differ!"
assert np.allclose(manual_scores.mean(), sklearn_scores.mean()), "means differ!"
print("\n✅ assert passed — our hand-rolled k-fold == cross_val_score, fold for fold.")

# every row was tested exactly once -> the test folds partition all 20 rows
all_test = sorted(idx for _, te in kf.split(X_dd) for idx in te)
assert all_test == list(range(20))
print("✅ every one of the 20 rows was in the test set exactly once.")

The two arrays are **identical fold-for-fold** — because `cross_val_score` *is* that loop, just packaged. Three things the proof nails down:

1. **Each fold trains on k−1 folds and tests on the 1 held-out fold** — exactly the rotation in the diagram.
2. **Every row is tested exactly once** (the final assert): the k test folds *partition* the dataset, so no row is wasted and none is double-counted.
3. **The reported number is the mean of the k fold scores** — five experiments collapsed into one honest estimate.

> ⚠️ **Don't fit anything on the full data before the split.** In the loop above, a *fresh* model is fit **inside** each fold on that fold's training rows only. The same rule applies to preprocessing — a scaler or encoder must be fit on each fold's *training* portion, never on the whole dataset, or information from the test fold leaks into training and your score becomes a fantasy. That's exactly why §8's rule #2 says *"put preprocessing inside the `Pipeline`"*: a `Pipeline` re-fits the scaler per fold automatically. (Debug Exercise 4 and NB 16 drill this leakage trap in depth.)

## 9. Learning curves — would more data help?

In [ ]:
LearningCurveDisplay.from_estimator(
    logreg, X, y, cv=5, scoring="roc_auc",
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1,
    score_type="both", std_display_style="fill_between",
)
plt.title("Learning curve — does more data help?");

**Reading a learning curve:**

- **Train and CV both low, close together** → underfitting (model too simple). More data won't help much; you need a *better* model.
- **Train high, CV much lower, gap shrinking with more data** → overfitting that's healing. Get more data.
- **Train high, CV high, gap small** → good fit. More data gives diminishing returns.

> 🎯 **The single most useful diagnostic plot.** Run it before spending a week labelling more data — sometimes you'll find out you don't need it.

## 🧪 Practice exercises

### Exercise 1 — ⭐ Cost-optimal threshold

Suppose `COST_FP = 20` and `COST_FN = 100`. Find the threshold that minimises total cost on the test set for the **Random Forest** model. By how much does it differ from threshold 0.5?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
COST_FP, COST_FN = 20, 100
thresholds = np.linspace(0.05, 0.95, 91)
costs = []
for t in thresholds:
    cm_ = confusion_matrix(y_te, (p_rf >= t).astype(int))
    costs.append(cm_[0,1]*COST_FP + cm_[1,0]*COST_FN)
best_t = thresholds[int(np.argmin(costs))]
print(f"Cost-optimal threshold = {best_t:.2f}")
print(f"Cost at 0.5            = ${costs[np.argmin(np.abs(thresholds-0.5))]:,}")
print(f"Cost at optimal        = ${min(costs):,}")
```

The optimal threshold shifts based on the cost ratio. With FN much more expensive than FP, the optimum is below 0.5 — you'd rather over-flag than miss.
</details>

### Exercise 2 — ⭐⭐ Compare AUCs with a confidence interval

Compute the ROC AUC for LR and RF on the test set, and a *95% confidence interval* for each by bootstrap-resampling the test set 500 times.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def bootstrap_auc(p, y, n_boot=500, rng=np.random.default_rng(0)):
    aucs = []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        fpr, tpr, _ = roc_curve(y[idx], p[idx])
        aucs.append(auc(fpr, tpr))
    return np.percentile(aucs, [2.5, 97.5])

for name, p in [("LR", p_lr), ("RF", p_rf)]:
    auc_val = auc(*roc_curve(y_te, p)[:2])
    lo, hi = bootstrap_auc(p, y_te)
    print(f"{name}: AUC = {auc_val:.3f}  (95% CI: [{lo:.3f}, {hi:.3f}])")
```

If the CIs overlap heavily, the difference between models is within noise — don't pick one over the other on a single test set.
</details>

### Exercise 3 — ⭐⭐ Calibration on LR

Repeat the calibration analysis (reliability diagram + Brier score) for the **logistic-regression** model. Is it better or worse calibrated than the RF? Why might that be?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
disp = CalibrationDisplay.from_predictions(y_te, p_lr, n_bins=10,
                                           strategy="quantile", name="LR")
disp.ax_.set_title("Reliability diagram — Logistic Regression")
plt.show()

print(f"Brier (LR)              : {brier_score_loss(y_te, p_lr):.3f}")
print(f"Brier (RF uncalibrated) : {brier_score_loss(y_te, p_rf):.3f}")
print(f"Brier (RF calibrated)   : {brier_score_loss(y_te, p_rf_cal):.3f}")
```

Logistic regression is usually well-calibrated *by construction* — it optimises the log-likelihood, which is a proper scoring rule. Random Forests don't, so they need post-hoc calibration to produce trustworthy probabilities.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The cell below tries to evaluate a model with cross-validation but mixes train and test data via a leaky preprocessing pipeline. Find the leak.

```python
from sklearn.preprocessing import StandardScaler

# Buggy: fits the scaler on ALL the data first
scaler = StandardScaler().fit(X.select_dtypes(include="number"))
X_scaled = scaler.transform(X.select_dtypes(include="number"))

model = LogisticRegression(max_iter=500)
scores = cross_val_score(model, X_scaled, y, cv=5, scoring="accuracy")
print(scores.mean())
```

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
# Your fixed version  👇


<details>
<summary>💡 <b>Solution</b></summary>

The scaler was fit on **all** of `X` — including the data that ends up in the CV test folds. The model then sees information from the held-out folds via the scale parameters.

```python
pipe = Pipeline([("scaler", StandardScaler()),
                  ("model",  LogisticRegression(max_iter=500))])
scores = cross_val_score(pipe, X.select_dtypes(include="number"), y, cv=5,
                          scoring="accuracy")
print(scores.mean())
```

**The discipline.** Always wrap preprocessing in a `Pipeline`. Cross-validation then fits the scaler on each fold's *training* portion only — and the score you report is honest.

This is the single most common source of "too-good-to-be-true" ML results in production.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — ⭐⭐⭐ F-beta — when precision and recall aren't equally important

Implement `f_beta(precision, recall, beta)` from scratch and compute F-0.5 (precision-favouring) and F-2 (recall-favouring) for the LR model at threshold 0.5. Comment on which one you'd use for churn.


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.metrics import precision_recall_fscore_support

def f_beta(p, r, beta):
    if p == r == 0: return 0.0
    b2 = beta * beta
    return (1 + b2) * p * r / (b2 * p + r)


y_pred_lr = (p_lr >= 0.5).astype(int)
prec, rec, _, _ = precision_recall_fscore_support(y_te, y_pred_lr, average="binary", pos_label=1, zero_division=0)
print(f"Precision = {prec:.3f}, Recall = {rec:.3f}")
print(f"F0.5      = {f_beta(prec, rec, 0.5):.3f}   (precision-favouring)")
print(f"F1        = {f_beta(prec, rec, 1.0):.3f}")
print(f"F2        = {f_beta(prec, rec, 2.0):.3f}   (recall-favouring)")
```

**For churn, F2 is usually right.** Missing a churner is much more
expensive than a false alarm — the customer is gone forever, but a
false alarm costs you maybe 10 minutes of CS time. Always pick a
metric whose β reflects the business cost ratio.

</details>

### Stretch exercise B — ⭐⭐⭐ Permutation feature importance

Compute **permutation feature importance** for the random forest using `sklearn.inspection.permutation_importance`. Plot a horizontal bar chart of the importances and discuss which features drive the predictions.


In [ ]:
# Your code here  👇
from sklearn.inspection import permutation_importance


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.inspection import permutation_importance

perm = permutation_importance(rforest, X_te, y_te, n_repeats=10,
                               random_state=RANDOM_STATE, n_jobs=-1)
feature_names = rforest.named_steps["prep"].get_feature_names_out()
order = np.argsort(perm.importances_mean)

fig, ax = plt.subplots(figsize=(8, 5))
pretty = [n.replace("num__","").replace("cat__plan_","plan=") for n in feature_names]
ax.barh(np.array(pretty)[order], perm.importances_mean[order],
         xerr=perm.importances_std[order], color="#4C72B0", edgecolor="black")
ax.set_title("Permutation feature importance (random forest)")
ax.set_xlabel("decrease in score")
plt.tight_layout(); plt.show()
```

**Permutation > built-in importance** in two ways. (1) It's
model-agnostic — works for any estimator, not just trees. (2)
It's measured *on a hold-out set*, so it can't confuse "the model
looked at this feature" with "the feature was actually predictive
out of sample".

</details>

### Stretch exercise C — ⭐⭐⭐ Pick a threshold to minimise expected cost

Most production classifiers don't optimise accuracy — they optimise **expected business cost**, where a false positive and a false negative carry different prices.

Suppose:

- A false positive costs **\$1** (we annoyed a happy customer with a retention call).
- A false negative costs **\$20** (we lost a customer we could have saved).

Given the predicted probabilities `y_score` and ground truth `y_true` below, find the threshold in `[0, 1]` that minimises the **total expected cost** on this dataset, and report the confusion matrix at that threshold.

Use a coarse grid of 101 candidate thresholds (`np.linspace(0, 1, 101)`).

In [ ]:
# Your code here  👇
import numpy as np
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(0)
y_true  = rng.integers(0, 2, size=500)
y_score = np.clip(0.1 + 0.7*y_true + 0.2*rng.standard_normal(500), 0, 1)

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(0)
y_true  = rng.integers(0, 2, size=500)
y_score = np.clip(0.1 + 0.7*y_true + 0.2*rng.standard_normal(500), 0, 1)

COST_FP, COST_FN = 1.0, 20.0

best_thr, best_cost = None, np.inf
for thr in np.linspace(0, 1, 101):
    y_pred = (y_score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    cost = fp * COST_FP + fn * COST_FN
    if cost < best_cost:
        best_cost, best_thr = cost, thr

print(f"best threshold = {best_thr:.2f}  expected cost = ${best_cost:.0f}")
y_pred = (y_score >= best_thr).astype(int)
print(confusion_matrix(y_true, y_pred, labels=[0,1]))
```

**Reasoning.** Two things to internalise. (1) Default 0.5 thresholds are a *convention*, not an optimum — once you write down the actual cost of FP vs FN, the right threshold is rarely 0.5. Here it shifts downwards because a missed positive (FN) is 20× more expensive than a false alarm. (2) The grid-search approach scales fine on any reasonable dataset and avoids the trap of analytic shortcuts that assume class-balanced costs. For a smoother result, use the **precision-recall curve** (`precision_recall_curve`) and pick the threshold algebraically — that's what production calibration code typically does.
</details>

### Stretch exercise D — ⭐⭐⭐ Compare two models with PR curves

Fit a `LogisticRegression` and a `RandomForestClassifier` on the breast-cancer dataset, then plot their **precision-recall curves on the same axes** and report **average precision (AP)** for each.

Why PR rather than ROC? On imbalanced or 'positives are rare' problems, PR is a much better lens — ROC can look good even when precision is terrible.

In [ ]:
# Your code here  👇
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import PrecisionRecallDisplay
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

models = {
    "logreg": LogisticRegression(max_iter=10_000).fit(X_tr, y_tr),
    "rf":     RandomForestClassifier(n_estimators=200, random_state=0).fit(X_tr, y_tr),
}

fig, ax = plt.subplots(figsize=(6, 4))
for name, m in models.items():
    PrecisionRecallDisplay.from_estimator(m, X_te, y_te, name=name, ax=ax)
ax.set_title("PR curves — breast cancer test set");
```

**Reasoning.** Average precision (AP) is the area under the PR curve and the single-number PR-equivalent of ROC's AUC. Unlike accuracy, AP doesn't reward a 'predict the majority class' model on imbalanced data. Two stylistic notes. (1) Always set `stratify=y` in your train/test split for classification — otherwise rare classes end up unevenly distributed and your test set becomes a different problem from your training set. (2) `PrecisionRecallDisplay.from_estimator` computes the curve, puts recall on the x-axis (the convention), and writes the AP into the legend — one line per model, no manual bookkeeping.
</details>

## 🎁 Bonus mini-project — A model-comparison report

Build a function `compare_models(models, X, y)` that:

1. For each named model, runs 5-fold stratified CV with `roc_auc` scoring.
2. Returns a DataFrame with columns `mean_auc`, `std_auc`, `min_auc`, `max_auc`.
3. Sorts by `mean_auc` descending.

Try it on LR, RF, and a default `GradientBoostingClassifier`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.ensemble import GradientBoostingClassifier

def compare_models(models: dict, X, y, cv_splits=5, scoring="roc_auc"):
    rows = []
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        rows.append({
            "model": name,
            "mean":  scores.mean(),
            "std":   scores.std(),
            "min":   scores.min(),
            "max":   scores.max(),
        })
    return pd.DataFrame(rows).set_index("model").sort_values("mean", ascending=False).round(3)


models = {
    "LR":  Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=500))]),
    "RF":  Pipeline([("prep", prep), ("m", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))]),
    "GBM": Pipeline([("prep", prep), ("m", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
}
# Stretch exercise D reassigned X, y to the breast-cancer dataset;
# restore the churn DataFrame so `prep` (which selects columns by name) works.
X = df.drop(columns=["churned"])
y = df["churned"]
print(compare_models(models, X, y))
```

This is the artefact you keep around: a 30-second model-comparison run that you re-execute every time you tweak a feature or a hyperparameter.
</details>

## 🧠 Key takeaways

1. **Accuracy alone is a trap.** Read the confusion matrix; assign costs to the cells.
2. **The decision threshold is a business knob**, not a model property. Optimise it on real cost.
3. **ROC AUC** for balanced problems, **PR AUC** for rare positives.
4. **Probabilities ≠ rankings.** Use a reliability diagram + Brier score; calibrate if needed.
5. **Stratified k-fold** + `Pipeline` is the leak-proof CV recipe.
6. **Learning curves** tell you whether more data or a better model is the right next step.
7. Every model report should include: confusion matrix at a chosen threshold, ROC + PR curves, calibration, CV mean ± std, and the cost-optimal threshold.

## ✅ Self-assessment

- [ ] Read a confusion matrix and explain its four cells in cost units
- [ ] Slide the decision threshold to balance precision and recall
- [ ] Plot ROC and PR curves and explain when each is right
- [ ] Draw a reliability diagram and interpret it
- [ ] Apply post-hoc calibration with `CalibratedClassifierCV`
- [ ] Run leak-proof cross-validation with `Pipeline`
- [ ] Read a learning curve and decide if more data would help

## 🚀 Next step

Continue with **Notebook 16 — Feature Engineering**, where the leaks you just learned to avoid get turned into a positive discipline: building features that *actually help* the model.